In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# =========================
# STEP 1: IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

# =========================
# STEP 2: LOAD CLEANED DATA
# =========================
df = pd.read_csv("/content/drive/MyDrive/cricket_dataset_cleaned_preprocessed.csv",low_memory=False)

print("Dataset Shape:", df.shape)

# =========================
# STEP 3: BASIC SANITY CHECK
# =========================
print("\nMissing values (total):", df.isna().sum().sum())
print("\nSample columns:", df.columns[:10].tolist())

# =========================
# STEP 4: FEATURE ENGINEERING
# =========================

df_fe = df.copy()

# -------------------------
# 1. Strike Rate
# -------------------------
df_fe['strike_rate'] = np.where(
    df_fe['batter_balls'] > 0,
    (df_fe['batter_runs'] / df_fe['batter_balls']) * 100,
    0
)

# -------------------------
# 2. Balls per Run
# -------------------------
df_fe['balls_per_run'] = np.where(
    df_fe['batter_runs'] > 0,
    df_fe['batter_balls'] / df_fe['batter_runs'],
    0
)

# -------------------------
# 3. Team Run Rate
# -------------------------
df_fe['run_rate'] = np.where(
    df_fe['overs'] > 0,
    df_fe['team_runs'] / df_fe['overs'],
    0
)

# -------------------------
# 4. Wickets per Over
# -------------------------
df_fe['wickets_per_over'] = np.where(
    df_fe['overs'] > 0,
    df_fe['team_wicket'] / df_fe['overs'],
    0
)

# -------------------------
# 5. Remaining Wickets
# -------------------------
df_fe['remaining_wickets'] = 10 - df_fe['team_wicket']

# -------------------------
# 6. Powerplay Indicator
# -------------------------
df_fe['is_powerplay'] = (df_fe['over'] <= 6).astype(int)

# -------------------------
# 7. Death Over Indicator
# -------------------------
df_fe['is_death_over'] = (df_fe['over'] >= 16).astype(int)

# -------------------------
# 8. Pressure Index
# -------------------------
df_fe['pressure_index'] = np.where(
    df_fe['team_balls'] > 0,
    df_fe['team_wicket'] / df_fe['team_balls'],
    0
)

# =========================
# STEP 5: VERIFY NEW FEATURES
# =========================
new_features = [
    'strike_rate', 'balls_per_run', 'run_rate',
    'wickets_per_over', 'remaining_wickets',
    'is_powerplay', 'is_death_over', 'pressure_index'
]

print("\n✅ Feature Engineering Completed")
print("New features added:", new_features)

print("\nSample of engineered data:")
df_fe[new_features].head()


Dataset Shape: (278205, 58)

Missing values (total): 2996189

Sample columns: ['match_id', 'date', 'match_type', 'event_name', 'innings', 'batting_team', 'bowling_team', 'over', 'ball', 'ball_no']

✅ Feature Engineering Completed
New features added: ['strike_rate', 'balls_per_run', 'run_rate', 'wickets_per_over', 'remaining_wickets', 'is_powerplay', 'is_death_over', 'pressure_index']

Sample of engineered data:


,strike_rate,balls_per_run,run_rate,wickets_per_over,remaining_wickets,is_powerplay,is_death_over,pressure_index
0,0.0,0.0,0.05,0.0,10.0,1,0,0.0
1,0.0,0.0,0.05,0.0,10.0,1,0,0.0
2,0.0,0.0,0.10,0.0,10.0,1,0,0.0
3,0.0,0.0,0.10,0.0,10.0,1,0,0.0
4,0.0,0.0,0.10,0.0,10.0,1,0,0.0


In [14]:
# =========================
# STEP 1: IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =========================
# STEP 2: LOAD CLEANED DATA
# =========================
df = pd.read_csv("/content/drive/MyDrive/cricket_dataset_cleaned_preprocessed.csv", low_memory=False)
print("Dataset Shape:", df.shape)

# =========================
# STEP 3: FIX DATA TYPES
# =========================
# Convert date
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Numeric columns
num_cols = [
    'innings', 'over', 'ball', 'ball_no', 'bat_pos',
    'balls_faced', 'runs_batter', 'runs_extras',
    'runs_total', 'runs_bowler', 'runs_not_boundary',
    'team_runs', 'team_balls', 'team_wicket',
    'balls_per_over', 'overs', 'batter_runs', 'batter_balls',
    'bowler_wicket'
]

# Convert to numeric safely
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# =========================
# STEP 4: HANDLE MISSING VALUES
# =========================
# Fill numeric NaNs with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical NaNs with 'None'
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna('None')

print("✔ Missing values handled")

# =========================
# STEP 5: FEATURE ENGINEERING
# =========================
df['strike_rate'] = np.where(df['batter_balls'] > 0,
                             (df['batter_runs'] / df['batter_balls']) * 100, 0)
df['balls_per_run'] = np.where(df['batter_runs'] > 0,
                               df['batter_balls'] / df['batter_runs'], 0)
df['run_rate'] = np.where(df['overs'] > 0,
                          df['team_runs'] / df['overs'], 0)
df['wickets_per_over'] = np.where(df['overs'] > 0,
                                  df['team_wicket'] / df['overs'], 0)
df['remaining_wickets'] = 10 - df['team_wicket']
df['is_powerplay'] = (df['over'] <= 6).astype(int)
df['is_death_over'] = (df['over'] >= 16).astype(int)
df['pressure_index'] = np.where(df['team_balls'] > 0,
                                df['team_wicket'] / df['team_balls'], 0)

print("✅ Feature engineering done")

# =========================
# STEP 6: DROP HIGH-NA / IRRELEVANT COLUMNS
# =========================
drop_cols = [
    'Unnamed: 0', 'extra_type', 'wicket_kind', 'player_out', 'fielders',
    'review_batter', 'team_reviewed', 'review_decision',
    'umpire', 'superover_winner', 'result_type', 'method',
    'match_id', 'player_of_match', 'match_won_by'
]

df.drop(columns=drop_cols, inplace=True, errors='ignore')
print("✔ Irrelevant/high-NaN columns dropped")

# =========================
# STEP 7: PREPARE FEATURES & TARGET
# =========================
target_col = 'win_outcome'  # Change if predicting something else

# Fill any remaining numeric NaNs in engineered features
engineered_numeric = ['strike_rate', 'balls_per_run', 'run_rate',
                      'wickets_per_over', 'remaining_wickets',
                      'is_powerplay', 'is_death_over', 'pressure_index']

for col in engineered_numeric:
    df[col] = df[col].fillna(0)

# Identify final numeric columns
numeric_cols += engineered_numeric

# Identify categorical columns (excluding target)
cat_cols = df.select_dtypes(include='object').columns.tolist()
if target_col in cat_cols:
    cat_cols.remove(target_col)

# =========================
# STEP 8: ONE-HOT ENCODE CATEGORICALS
# =========================
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# =========================
# STEP 9: SCALE NUMERIC FEATURES
# =========================
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

# =========================
# STEP 10: TRAIN-TEST SPLIT
# =========================
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.nunique() <= 10 else None
)

print("✅ Dataset ready for model training")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)



Dataset Shape: (278205, 58)
✔ Missing values handled
✅ Feature engineering done
✔ Irrelevant/high-NaN columns dropped
✅ Dataset ready for model training
X_train shape: (222564, 2152)
X_test shape: (55641, 2152)
y_train shape: (222564,)
y_test shape: (55641,)


In [17]:
df.to_csv("/content/drive/MyDrive/cricket_dataset_final_ready.csv", index=False)
print("✅ Dataset cleaned and saved as 'cricket_dataset_final.csv'")

✅ Dataset cleaned and saved as 'cricket_dataset_final.csv'
